In [1]:
!git clone https://github.com/keiranoluv/final_project
!git clone https://github.com/PaddlePaddle/PaddleOCR.git

Cloning into 'final_project'...
remote: Enumerating objects: 38, done.
remote: Counting objects: 100% (38/38), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 38 (delta 8), reused 32 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (38/38), 18.64 KiB | 1004.00 KiB/s, done.
Resolving deltas: 100% (8/8), done.
Cloning into 'PaddleOCR'...
remote: Enumerating objects: 353017, done.
remote: Counting objects: 100% (1268/1268), done.
remote: Compressing objects: 100% (171/171), done.
remote: Total 353017 (delta 1187), reused 1097 (delta 1097), pack-reused 351749 (from 3)
Receiving objects: 100% (353017/353017), 1.87 GiB | 33.90 MiB/s, done.
Resolving deltas: 100% (279322/279322), done.


In [2]:
%cd /kaggle/working/PaddleOCR

!python -m pip install -q -r requirements.txt
!python -m pip install -q paddlepaddle-gpu==3.3.0 \
  -i https://www.paddlepaddle.org.cn/packages/stable/cu126/ \
  --no-deps

/kaggle/working/PaddleOCR
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 344.7/344.7 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 61.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 GB 701.2 kB/s eta 0:00:00


In [3]:
import csv
from pathlib import Path

DATASET_ROOT = Path("/kaggle/input/datasets/zephyrvn/nlp-project-data/MTHv2_processed/MTHv2")
OUTPUT_ROOT = Path("/kaggle/working/mthv2_labels")

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

for split in ["train", "val", "test"]:
    src = DATASET_ROOT / f"{split}.tsv"
    dst = OUTPUT_ROOT / f"{split}.txt"

    count = 0

    with src.open("r", encoding="utf-8") as fin, \
         dst.open("w", encoding="utf-8") as fout:

        reader = csv.DictReader(fin, delimiter="\t")

        for row in reader:
            fout.write(f'{row["image_path"]}\t{row["text"]}\n')
            count += 1

    print(f"{split}: {count:,} samples -> {dst}")

train: 72,563 samples -> /kaggle/working/mthv2_labels/train.txt
val: 7,753 samples -> /kaggle/working/mthv2_labels/val.txt
test: 25,262 samples -> /kaggle/working/mthv2_labels/test.txt


In [4]:
import csv
from pathlib import Path

src = Path("/kaggle/input/datasets/zephyrvn/vocabylary-expanded/train_rare_oversampled.tsv")
dst = Path("/kaggle/working/mthv2_labels/train_rare_oversampled.txt")

count = 0

with src.open("r", encoding="utf-8") as fin, \
     dst.open("w", encoding="utf-8") as fout:

    reader = csv.DictReader(fin, delimiter="\t")

    for row in reader:
        fout.write(f'{row["image_path"]}\t{row["text"]}\n')
        count += 1

print(f"B3 train: {count:,} samples -> {dst}")

B3 train: 120,093 samples -> /kaggle/working/mthv2_labels/train_rare_oversampled.txt


## B3 — Rare-Character-Aware Fine-Tuning

### 1. Motivation

Phân tích lỗi của B2 trên tập validation cho thấy mô hình vẫn có xu hướng nhầm các ký tự ít xuất hiện trong tập huấn luyện sang các ký tự phổ biến hơn.

Kết quả phân tích substitution trên validation:

* Tổng số substitution: **1,822**
* Ground-truth thuộc nhóm ký tự hiếm (`train_freq <= 20`): **48.52%**
* Prediction có tần suất trong train cao hơn ground-truth: **65.97%**
* Substitution theo hướng rare → common: **43.74%**
* Substitution theo hướng common → rare: **10.87%**
* Weighted mean log2 frequency ratio: **1.637**

Kết quả cho thấy lỗi substitution có tính bất đối xứng và thường nghiêng về các class xuất hiện nhiều hơn trong training data.

Do đó, sau khi B2 giải quyết vấn đề vocabulary coverage bằng expanded vocabulary, B3 tập trung vào vấn đề tiếp theo: **class-frequency imbalance đối với các ký tự hiếm**.

---

### 2. Hypothesis

Expanded vocabulary giúp model có khả năng biểu diễn các ký tự lịch sử và ký tự hiếm, nhưng không đảm bảo model học được representation đủ tốt cho các class có ít training examples.

Giả thuyết của B3:

> Tăng mức độ xuất hiện của các training samples chứa ký tự hiếm sẽ giúp giảm bias của mô hình về các ký tự phổ biến và giảm lỗi rare-to-common substitution.

---

### 3. Rare-character definition

Tần suất ký tự được tính trực tiếp trên training set.

Trong B3, một ký tự được xem là rare nếu:

```text
train_frequency <= 50
```

Threshold 50 được chọn sau khi phân tích validation, trong đó xu hướng rare-to-common vẫn thể hiện rõ khi mở rộng nhóm low-frequency characters.

---

### 4. Oversampling strategy

Không thay đổi ảnh, label hoặc validation/test set.

Mỗi training line được kiểm tra xem có chứa ít nhất một rare character hay không.

Sampling strategy:

```text
Normal training line     → 1 copy
Rare-character line      → 3 copies total
```

Tức là mỗi rare-character line được bổ sung thêm 2 bản sao.

Không thực hiện augmentation mới trong B3 để cô lập tác động của rare-character-aware sampling.

---

### 5. Resulting training set

Training set gốc:

```text
72,563 lines
```

Số line chứa ít nhất một rare character:

```text
23,765 lines
32.75% of training set
```

Số duplicate được bổ sung:

```text
47,530 lines
```

Training manifest B3:

```text
120,093 lines
```

Dataset expansion:

```text
1.655×
```

File:

```text
train_rare_oversampled.tsv
```

---

### 6. Training protocol

B3 tiếp tục fine-tune từ **B2 best checkpoint**, không khởi tạo lại từ pretrained PP-OCRv5.

```text
PP-OCRv5 pretrained
        ↓
B1 — MTHv2 fine-tuning
        ↓
B2 — Expanded vocabulary
        ↓
B3 — Rare-character-aware fine-tuning
```

B3 giữ nguyên:

* PP-OCRv5_server_rec architecture
* Expanded vocabulary của B2
* Image preprocessing
* Validation set
* Evaluation metric
* Unicode normalization
* Recognition pipeline

Thay đổi chính duy nhất:

```text
Training sampling distribution
```

Training manifest:

```text
train_rare_oversampled.tsv
```

Validation manifest:

```text
val.tsv
```

Test manifest:

```text
test.tsv
```

Test set không được sử dụng để lựa chọn hyperparameter hoặc quyết định checkpoint của B3.

---

### 7. Initial training configuration

B3 là corrective fine-tuning từ B2 nên sử dụng learning rate thấp và số epoch ngắn hơn quá trình fine-tuning chính.

Initial configuration:

```text
Initialization      : B2 best checkpoint
Rare threshold      : 50
Oversample factor   : 3
Epochs              : 3–5
Vocabulary          : B2 expanded vocabulary
Validation          : original val.tsv
Test                : untouched test.tsv
```

Best checkpoint được lựa chọn dựa trên validation performance.

---

### 8. Evaluation

Trước tiên so sánh B2 và B3 trên validation set:

```text
B2 validation CER
vs.
B3 validation CER
```

Ngoài CER và exact-match accuracy, chạy lại rare-character bias analysis.

Các chỉ số chính:

```text
CER
Exact-match accuracy
Rare → common substitution count
Rare → common substitution percentage
Prediction-more-frequent percentage
Weighted mean log2 frequency ratio
```

B3 được xem là có tác động đúng giả thuyết nếu đồng thời quan sát được:

```text
CER ↓
rare → common substitutions ↓
```

Nếu B3 cải thiện trên validation, checkpoint và toàn bộ preprocessing được freeze trước khi chạy test cuối cùng.

---

### 9. Interpretation

B3 nhằm xử lý lỗi ở mức training distribution thay vì sửa trực tiếp prediction bằng hard-coded post-processing rules.

Điều này giúp phân biệt ba vấn đề:

```text
B1 → general domain adaptation
B2 → vocabulary / character coverage
B3 → rare-character class imbalance
```

Historical glyph variants và codepoint-equivalent forms vẫn được phân tích riêng vì các confusion này không nhất thiết xuất phát từ class imbalance.


In [5]:
%cd /kaggle/working/PaddleOCR

!python -m paddle.distributed.launch \
  --gpus "0,1" \
  tools/train.py \
  -c configs/rec/PP-OCRv5/PP-OCRv5_server_rec.yml \
  -o \
  Global.pretrained_model=/kaggle/input/models/zephyrvn/b2-best-checkpoint/other/default/1/best_model_b2_open_vocab.pdparams \
  Global.character_dict_path=/kaggle/input/datasets/zephyrvn/vocabylary-expanded/ppocrv5_mthv2_expanded.txt \
  Global.epoch_num=10 \
  Global.save_model_dir=/kaggle/working/final_project/outputs/B3_rare_char_ft_10ep \
  Global.eval_batch_step="[0,100]" \
  Optimizer.lr.learning_rate=0.00005 \
  Optimizer.lr.warmup_epoch=0 \
  Train.loader.batch_size_per_card=64 \
  Train.sampler.first_bs=64 \
  Train.dataset.data_dir=/kaggle/input/datasets/zephyrvn/nlp-project-data/MTHv2_processed/MTHv2 \
  Train.dataset.label_file_list='["/kaggle/working/mthv2_labels/train_rare_oversampled.txt"]' \
  Eval.dataset.data_dir=/kaggle/input/datasets/zephyrvn/nlp-project-data/MTHv2_processed/MTHv2 \
  Eval.dataset.label_file_list='["/kaggle/working/mthv2_labels/val.txt"]'

/kaggle/working/PaddleOCR
/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
LAUNCH INFO 2026-08-09 10:34:19,475 -----------  Configuration  ----------------------
LAUNCH INFO 2026-08-09 10:34:19,476 auto_cluster_config: 0
LAUNCH INFO 2026-08-09 10:34:19,476 auto_parallel_config: None
LAUNCH INFO 2026-08-09 10:34:19,476 auto_tuner_json: None
LAUNCH INFO 2026-08-09 10:34:19,476 devices: 0,1
LAUNCH INFO 2026-08-09 10:34:19,476 elastic_level: -1
LAUNCH INFO 2026-08-09 10:34:19,476 elastic_timeout: 30
LAUNCH INFO 2026-08-09 10:34:19,476 enable_gpu_log: True
LAUNCH INFO 2026-08-09 10:34:19,476 gloo_port: 6767
LAUNCH INFO 2026-08-09 10:34:19,476 host: None
LAUNCH INFO 2026-08-09 10:34:19,476 ips: None
LAUNCH INFO 2026-08-09 